# Libraries Import and Base Path initialization

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install ultralytics

In [1]:
import os, cv2, json, math
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import tensorflow as tf
import tensorflow_hub as hub
from ultralytics import YOLO
# import pytorch as torch

# from mmengine.config import Config
# from mmengine.registry import MODELS
# from mmengine.runner import load_checkpoint
# from mmaction.apis import init_recognizer

# # This function will now work correctly because we are running from the cloned directory
# from mmaction.utils import register_all_modules
# register_all_modules(init_default_scope=True) # We set the scope manually later

#Colab Base Path
# base_path = "/content/drive/MyDrive/SMT 6/CV/UAS"

#Local Base Path
base_path = ""

#Dataset Paths
# data_path = os.path.join(base_path, "match_videos")
data_path = os.path.join(base_path, "practice_videos")

# video_path = os.path.join(data_path, "Knee(Bryan) vs Double(Law) TWT 2024.mp4")
# video_path = os.path.join(data_path, "Knee(Bryan) vs Double(Law) 1 round.mp4")
video_path = os.path.join(data_path, "Bryan_L.mp4")

# annotation_path = os.path.join(base_path, "match_videos/Knee(Bryan) vs Double(Law) TWT 2024.json")
annotation_path = os.path.join(data_path, "Knee_reindexed.json")

output_dir = os.path.join(data_path, "frames")
kp_dir = os.path.join(data_path)

# video_path = os.path.join(base_path, "Bryan_2/Bryan_15_move_trimmed.mp4")
# annotation_path = os.path.join(base_path, "Bryan_2/Bryan_15_move_2.json")
# output_dir = os.path.join(base_path, "Bryan_2/frames")

# Load movenet^
movenet = hub.load("https://tfhub.dev/google/movenet/singlepose/lightning/4").signatures['serving_default']
# yolo = YOLO("yolo11s.pt")
# yolo = YOLO("yolo11m.pt")
# yolo = YOLO("yolo26s.pt")
yolo  = YOLO("runs/detect/practice/weights/best.pt")
yolo.to("mps")


# os.makedirs(output_dir, exist_ok=True)

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(96, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_runnin

# YOLO and ByteTrack character tracking

In [2]:
def yolo_tracking(image):
    #f"{output_dir}/frame_00191.jpg"
    result = yolo.track(image, tracker="bytetrack.yaml", persist=True, conf=0.5, iou=0.1, classes=[0])[0]
    # the results of yolo.track contains list of object per frame it tracks, for example image will have 1 object in the list
    # while video will have as many object in it as the video frames
    # we will access the first object as it is an image
    if result.boxes is None:
        return None

    if result.boxes.id is None:
        return None

    ids = result.boxes.id.cpu().numpy().astype(int) # convert boxes ids of selected object to numpy, then to int
    boxes = result.boxes.xyxy.cpu().numpy().astype(int) # convert boxes x1, y1, x2, y2 of selected object to numpy, then to int

    id_box_array = np.hstack((ids.reshape(-1, 1), boxes))
    # stack ids and boxes horizontally (it only accepts tuple so we encapsulate it with double ()

    # print("Results xyxy", result.boxes.xyxy) # print the x1, y1, x2, y2 from boxes of the object
    # print("Results id", result.boxes.id) # print the id from boxes of the object
    # print("id, box array", id_box_array, type(id_box_array))
    return id_box_array

def pad_box(h, w, box, padding=25):
    # box: [id, x1, y1, x2, y2]
    _, x1, y1, x2, y2 = box
    x1 = max(0, x1 - padding)
    y1 = max(0, y1 - padding)
    x2 = min(w, x2 + padding)
    y2 = min(h, y2 + padding)

    return np.array([_, x1, y1, x2, y2], dtype=int)
    
def crop_roi(frame, box):
    # box: [id, x1, y1, x2, y2]
    _, x1, y1, x2, y2 = box
    # print("x1: ", x1, "y1: ", y1, "x2: ", x2, "y2: ", y2)
    return frame[y1:y2, x1:x2]

def display_roi(player1_roi, player2_roi):
    fig, axes = plt.subplots(nrows = 1, ncols = 2, figsize = (4, 8))
    axes[0].imshow(player1_roi)
    axes[1].imshow(player2_roi)

    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

# Movenet Holistic Function

In [3]:
def run_movenet(image, input_size=192):
    image = tf.image.resize_with_pad(image, input_size, input_size)
    # image = tf.cast(image, dtype=tf.float32) # if using gpu ?? not sure
    image = tf.cast(image, dtype=tf.int32) # if using cpu
    image = tf.expand_dims(image, axis=0)

    output = movenet(image)
    # The output is a dictionary, and we need the keypoints from 'output_0'
    keypoints_list = output['output_0'].numpy()[0]
    # print("Image shape", image.shape)
    # print("Keypoints list shape", keypoints_list.shape)
    # print("Keypoints list [0] shape", keypoints_list[0].shape)

    return keypoints_list[0]

def denormalize_points(points, original_height, original_width, input_size=192):
    """
    Converts normalized keypoints or bounding boxes from MoveNet output
    back to original image coordinates.
    """
    scale = min(input_size / original_height, input_size / original_width)
    new_height = original_height * scale
    new_width = original_width * scale
    pad_y = (input_size - new_height) / 2
    pad_x = (input_size - new_width) / 2

    y, x, c = points
    x_abs = ((x * input_size) - pad_x) / scale
    y_abs = ((y * input_size) - pad_y) / scale
    return (int(y_abs), int(x_abs), c)

def normalize_points_to_full_frame(kp_array, box, full_height, full_width):
    _, x1, y1, x2, y2 = box
    roi_height = y2-y1
    roi_width = x2-x1

    kp_full = []
    for kp in kp_array:
        y_roi, x_roi, c = denormalize_points(kp, roi_height, roi_width, 192)
        y_full = (y_roi + y1) / full_height
        x_full = (x_roi + x1) / full_width
        kp_full.append([y_full, x_full, c])

    return np.array(kp_full)
        

def interpolate_points(player_kp):
    player_kp = np.array(player_kp)
    for kp in range(player_kp.shape[1]):
        for coord in range(player_kp.shape[2]):
            data = player_kp[:, kp, coord]
            nans = np.isnan(data) # nans mask example: [true, false, true] based on the positions
            if np.any(~nans):
                data[nans] = np.interp(np.flatnonzero(nans), np.flatnonzero(~nans), data[~nans])
            player_kp[:, kp, coord] = data
            print(data)
    return player_kp
# for kp in player1_kp:
#     print(kp)
#     y, x, c = denormalize_points(kp, original_height, original_width)

# Movenet Holistic Extraction

In [4]:
input_size = 192
cap = cv2.VideoCapture(video_path)
print(video_path, os.path.exists(video_path))

player1_kp = []
player2_kp = []
last_p1_box, last_p2_box = None, None
player1_box, player2_box, other_box = None, None, None
player1_kp_full, player2_kp_full, other_kp_full = None, None, None
frame_count = 0

def get_center(box):
    # Returns the (x, y) center of a bounding box [id, x1, y1, x2, y2]
    return ((box[1] + box[3]) / 2, (box[2] + box[4]) / 2)

def get_distance(box1, box2):
    c1 = get_center(box1)
    c2 = get_center(box2)
    return math.sqrt((c1[0] - c2[0])**2 + (c1[1] - c2[1])**2)

while cap.isOpened(): # read every single frame of the video
    ret, frame_bgr = cap.read()
    if not ret:
        print("End of video")
        break

    # frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    id_box_array = yolo_tracking(frame_bgr) # 2d array containing id_box from p1 and 2
    # print(id_box_array)

    if id_box_array is None or id_box_array.size == 0:
        player1_kp.append(np.full((17, 3), np.nan))
        player2_kp.append(np.full((17, 3), np.nan))
        frame_count += 1
        continue

    if frame_count == 0:
        original_height, original_width = frame_bgr.shape[:2]
        print("Original height, original_width", original_height, original_width)
        player1_id = id_box_array[0, 0]
        player2_id = id_box_array[1, 0]

    p1_exist = any(id_box_array[:, 0] == player1_id)
    p2_exist = any(id_box_array[:, 0] == player2_id)
    # other_mask = ~np.isin(id_box_array[:, 0], [player1_id, player2_id])
    # has_other = np.any(other_mask)

    # === PLAYER 1 TRACKING WITH SPATIAL MEMORY ===
    if p1_exist:
        # Normal tracking
        raw_p1_box = id_box_array[id_box_array[:, 0] == player1_id][0]
        last_p1_box = raw_p1_box # Save memory of last known location
    else:
        # Tracker lost ID. Use spatial memory to recover!
        raw_p1_box = None
        if id_box_array is not None and len(id_box_array) > 0 and last_p1_box is not None:
            closest_box = min(id_box_array, key=lambda b: get_distance(b, last_p1_box))
            
            # Distance threshold check (jump hasn't exceeded 200 pixels center-to-center)
            if get_distance(closest_box, last_p1_box) < 200:
                raw_p1_box = closest_box
                last_p1_box = raw_p1_box # Update memory
                player1_id = closest_box[0] # Override the global ID to heal the tracker!

    # Run MoveNet for Player 1
    if raw_p1_box is not None:
        player1_box_padded = pad_box(original_height, original_width, raw_p1_box)
        player1_roi = crop_roi(frame_bgr, player1_box_padded)
        player1_kp_raw = run_movenet(player1_roi, input_size)
        player1_kp_full = normalize_points_to_full_frame(player1_kp_raw, player1_box_padded, original_height, original_width)
        player1_kp.append(player1_kp_full)
    else:
        player1_kp.append(np.full((17, 3), np.nan))

    # === PLAYER 2 TRACKING WITH SPATIAL MEMORY ===
    if p2_exist:
        raw_p2_box = id_box_array[id_box_array[:, 0] == player2_id][0]
        last_p2_box = raw_p2_box
    else:
        raw_p2_box = None
        if id_box_array is not None and len(id_box_array) > 0 and last_p2_box is not None:
            # We filter out the box we already claimed for player 1 to avoid overlap
            available_boxes = [b for b in id_box_array if raw_p1_box is None or b[0] != raw_p1_box[0]]
            
            if len(available_boxes) > 0:
                closest_box = min(available_boxes, key=lambda b: get_distance(b, last_p2_box))
                if get_distance(closest_box, last_p2_box) < 200:
                    raw_p2_box = closest_box
                    last_p2_box = raw_p2_box
                    player2_id = closest_box[0]

    # Run MoveNet for Player 2
    if raw_p2_box is not None:
        player2_box_padded = pad_box(original_height, original_width, raw_p2_box)
        player2_roi = crop_roi(frame_bgr, player2_box_padded)
        player2_kp_raw = run_movenet(player2_roi, input_size)
        player2_kp_full = normalize_points_to_full_frame(player2_kp_raw, player2_box_padded, original_height, original_width)
        player2_kp.append(player2_kp_full)
    else:
        player2_kp.append(np.full((17, 3), np.nan))

    # if has_other:
    #     other_box = id_box_array[other_mask][0]
    #     other_box = pad_box(original_height, original_width, other_box)
    #     other_roi = crop_roi(frame_bgr, other_box)
    #     other_kp_raw = run_movenet(other_roi, input_size)
    #     other_kp_full = normalize_points_to_full_frame(other_kp_raw, other_box, original_height, original_width)


    # Visualization: draw player 1 (green) and player 2 (red)
    color_p1 = (0, 255, 0)  # green (B, G, R)
    color_p2 = (0, 0, 255)  # red
    
    # Boxes + IDs
    if player1_box_padded is not None:
        obj_id, x1, y1, x2, y2 = player1_box_padded
        cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), color_p1, thickness=2, lineType=cv2.LINE_AA)
        cv2.putText(frame_bgr, "ID: " + str(int(obj_id)), (x1, y1+10), cv2.FONT_HERSHEY_SIMPLEX,
                    fontScale=1, color=color_p1, thickness=1, lineType=cv2.LINE_AA)
    
    if player2_box_padded is not None:
        obj_id, x1, y1, x2, y2 = player2_box_padded
        cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), color_p2, thickness=2, lineType=cv2.LINE_AA)
        cv2.putText(frame_bgr, "ID: " + str(int(obj_id)), (x1, y1+10), cv2.FONT_HERSHEY_SIMPLEX,
                    fontScale=1, color=color_p2, thickness=1, lineType=cv2.LINE_AA)
    
    # Keypoints
    if player1_kp_full is not None:
        for kp in player1_kp_full:
            y, x, c = kp
            y, x = y * original_height, x * original_width
            cv2.circle(frame_bgr, (int(x), int(y)), 3, color_p1, thickness=2, lineType=cv2.LINE_AA)
    
    if player2_kp_full is not None:
        for kp in player2_kp_full:
            y, x, c = kp
            y, x = y * original_height, x * original_width
            cv2.circle(frame_bgr, (int(x), int(y)), 3, color_p2, thickness=2, lineType=cv2.LINE_AA)

    cv2.imshow('Process feed', frame_bgr)
    if cv2.waitKey(25) & 0xFF == ord('q'):
        break

    # frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    # plt.imshow(frame_rgb)
    # plt.show()

    frame_count += 1

cap.release()

player1_kp = interpolate_points(player1_kp)
player2_kp = interpolate_points(player2_kp)

np.save(os.path.join(kp_dir, "player1_kp"), player1_kp)
np.save(os.path.join(kp_dir, "player2_kp"), player2_kp)

practice_videos/Bryan_L.mp4 True

0: 416x640 2 fighters, 184.7ms
Speed: 5.4ms preprocess, 184.7ms inference, 35.4ms postprocess per image at shape (1, 3, 416, 640)
Original height, original_width 878 1352

0: 416x640 2 fighters, 18.0ms
Speed: 1.8ms preprocess, 18.0ms inference, 5.4ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 2 fighters, 11.0ms
Speed: 1.5ms preprocess, 11.0ms inference, 4.3ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 2 fighters, 10.1ms
Speed: 1.5ms preprocess, 10.1ms inference, 5.5ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 2 fighters, 13.7ms
Speed: 1.5ms preprocess, 13.7ms inference, 6.4ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 2 fighters, 12.7ms
Speed: 1.4ms preprocess, 12.7ms inference, 6.5ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 2 fighters, 13.0ms
Speed: 1.4ms preprocess, 13.0ms inference, 6.3ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 2 fighters, 12.

Sure! Here's a concise summary of everything we discussed, formatted in markdown for easy reference:

---

## 📚 Summary: MMAction2 Skeleton Dataset Format & Preparation Steps

Skeleton-based Action Recognition in MMAction2 doesn’t require splitting the original video, but it **does require splitting the keypoint data** into action-based segments.

---

### 🧬 Dataset Format Overview (`.pkl`)

```python
{
  "split": {
    "train": ["clip1", "clip2", ...],
    "val": ["clip7", "clip8", ...],
    ...
  },
  "annotations": [
    {
      "frame_dir": "clip1",
      "label": 0,
      "img_shape": (1080, 1920),
      "original_shape": (1080, 1920),
      "total_frames": 87,
      "keypoint": np.ndarray([M, T, V, C]),
      "keypoint_score": np.ndarray([M, T, V])
    },
    ...
  ]
}
```

- **`frame_dir`**: Unique name for each clip
- **`label`**: Action class (int)
- **`img_shape` & `original_shape`**: Optional frame resolution
- **`total_frames`**: Frames in the segment
- **`keypoint`**: Shape `[M x T x V x C]` (people, frames, joints, coords)
- **`keypoint_score`**: Confidence for each keypoint `[M x T x V]`

---

### ⚙️ Steps to Prepare from a Long Video

If you already extracted full video keypoints:

1. **Use Annotations**  
   Get frame ranges for each action from your annotation file.

2. **Slice Keypoint Arrays**  
   Extract each action clip from the full keypoint array using its frame indices.

3. **Assign Clip Identifiers**  
   Name each segment like `clip001`, `clip002`, etc.

4. **Group into Splits**  
   Organize clip names into `'train'`, `'val'`, etc. inside the `split` dictionary.

5. **Build Annotations List**  
   For each clip, create a dictionary with all required fields and add it to `annotations`.

6. **Save to Pickle**  
   Combine `split` and `annotations` into a Python dict and save as `.pkl`.

---

Want me to build a sample Python script to help automate these steps? Happy to dive in! 💻

# Prepare dataloader

In [5]:
with open(annotation_path) as f:
    annotations = json.load(f)

sequences_id = list(annotations.keys())[0]
data = annotations[sequences_id]
print(sequences_id, data)
# for sequence_id, data in annotations.items():
#     print(sequence_id)
#     print(data)

FileNotFoundError: [Errno 2] No such file or directory: 'practice_videos/Knee_reindexed.json'

# Prepare STGCN++ model

In [ ]:
config_file = "https://github.com/open-mmlab/mmaction2/blob/main/configs/skeleton/stgcnpp/stgcnpp_8xb16-joint-motion-u100-80e_ntu60-xsub-keypoint-2d.py"
checkpoint = "https://download.openmmlab.com/mmaction/v1.0/skeleton/stgcnpp/stgcnpp_8xb16-joint-motion-u100-80e_ntu60-xsub-keypoint-2d/stgcnpp_8xb16-joint-motion-u100-80e_ntu60-xsub-keypoint-2d_20221228-19a34aba.pth"

: 